In [0]:
!nc -vz pub.worldb.dedyn.io 9000

In [0]:
!ls -la /Volumes/workspace/default/on_premises_certificates/postgres

In [0]:
%sh
CERT_DIR=/Volumes/workspace/default/on_premises_certificates/postgres

ls -la "$CERT_DIR"

file "$CERT_DIR"/postgres-client.p12

for cert in ca.crt client.crt; do
  echo "===== $cert ====="
  openssl x509 \
    -in "$CERT_DIR/$cert" \
    -noout \
    -subject \
    -issuer \
    -serial \
    -dates \
    -fingerprint \
    -sha256
done

In [0]:
for scope in dbutils.secrets.listScopes():
    print(f"\nScope: {scope.name}")
    for secret in dbutils.secrets.list(scope.name):
        print(f"  {secret.key}")

dbutils.secrets.list("on-premises-integration")    

In [0]:
from pathlib import Path

postgres_host = "pub.worldb.dedyn.io"
postgres_port = 9000
postgres_database = "postgres"
postgres_user = "postgres"

certificate_dir = (
    Path("/Volumes/workspace/default/on_premises_certificates/postgres")
)
ca_certificate_path = certificate_dir / "ca.crt"
client_keystore_path = certificate_dir / "postgres-client.p12"

for certificate_path in (ca_certificate_path, client_keystore_path):
    if not certificate_path.is_file():
        raise FileNotFoundError(
            f"Missing PostgreSQL mTLS file: {certificate_path}"
        )

client_keystore_password = dbutils.secrets.get(
    scope="on-premises-integration",
    key="postgres-client-keystore-password",
)

postgres_password = dbutils.secrets.get(
    scope="on-premises-integration",
    key="postgres-password",
)

jdbc_url = (
    f"jdbc:postgresql://{postgres_host}:{postgres_port}/"
    f"{postgres_database}"
)

In [0]:
print(jdbc_url)
print(client_keystore_password)
print(ca_certificate_path)
!ls -la {ca_certificate_path}
print(client_keystore_path)
!ls -la {client_keystore_path}

# Debug

```shell
docker compose logs postgres
```
---
```shell
docker compose exec postgres openssl x509 \
  -in /var/lib/postgresql/tls/ca.crt \
  -noout -fingerprint -sha256
```

In [0]:
for scope in dbutils.secrets.listScopes():
    print(f"\nScope: {scope.name}")
    for secret in dbutils.secrets.list(scope.name):
        print(f"  {secret.key}")

dbutils.secrets.list("on-premises-integration")  

In [0]:
# NOTE: Client certificate authentication with PostgreSQL JDBC on Spark Connect
# is not supported due to filesystem access limitations. Using psycopg2 instead.
#
# UC Volumes have permissive file permissions (0777) that psycopg2 rejects for
# private keys. Copy the key to /tmp with restricted permissions.

try:
    import psycopg2
except ImportError:
    raise RuntimeError(
        "psycopg2 is not installed. Install it with: %pip install psycopg2-binary"
    )

import shutil
import os
import tempfile

# Create temp directory for certificates with proper permissions
temp_dir = tempfile.mkdtemp()
temp_key = os.path.join(temp_dir, "client.key")
temp_cert = os.path.join(temp_dir, "client.crt")
temp_ca = os.path.join(temp_dir, "ca.crt")

# Copy files and set restrictive permissions
shutil.copy(certificate_dir / "client.key", temp_key)
shutil.copy(certificate_dir / "client.crt", temp_cert)
shutil.copy(ca_certificate_path, temp_ca)
os.chmod(temp_key, 0o600)
os.chmod(temp_cert, 0o600)
os.chmod(temp_ca, 0o600)

# Server requires client certificate (mTLS)
# Use sslmode=require to provide client cert but skip server cert verification
connection_params = {
    "host": postgres_host,
    "port": postgres_port,
    "database": postgres_database,
    "user": postgres_user,
    #"password": client_keystore_password, # mTLS
    "password": postgres_password,
    "sslmode": "require",
    "sslcert": temp_cert,
    "sslkey": temp_key,
}

try:
    conn = psycopg2.connect(**connection_params)
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT
          current_user AS database_user,
          ssl,
          version AS tls_version,
          cipher AS tls_cipher,
          bits AS tls_bits,
          client_dn
        FROM pg_stat_ssl
        WHERE pid = pg_backend_pid()
    """)
    
    results = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    
    # Convert to pandas DataFrame for display
    import pandas as pd
    test_df = pd.DataFrame(results, columns=columns)
    display(test_df)
    
    cursor.close()
    conn.close()
    
finally:
    # Clean up temp files
    import shutil
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)